# Weight survey B — seed 123

Config: `natural,balanced,title_heavy`.


In [ ]:
import glob, json, shutil, subprocess, sys
from pathlib import Path
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('no GPU')
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory/1e9, 1), 'GB')


In [ ]:
DATA = Path('/kaggle/input/tiki-weight-ab-data2')
if not (DATA / 'units_A.parquet').exists():
    hits = sorted(Path('/kaggle/input').rglob('units_A.parquet'))
    assert hits, 'không thấy units_A.parquet'
    DATA = hits[0].parent
ROOT = Path('/kaggle/working/ab_data'); ROOT.mkdir(parents=True, exist_ok=True)
for ds in ('A', 'B'):
    d = ROOT / f'processed_{ds}'; d.mkdir(exist_ok=True)
    shutil.copy2(DATA / f'units_{ds}.parquet', d / 'units.parquet')
    shutil.copy2(DATA / f'eval_{ds}.parquet', ROOT / f'eval_{ds}.parquet')
    shutil.copy2(DATA / f'meta_{ds}.json', ROOT / f'meta_{ds}.json')

CODE = Path('/kaggle/working/code'); CODE.mkdir(exist_ok=True)
trainer = sorted(Path('/kaggle/input').rglob('train_weight_quota_gpu.py'))[0]
for name in ('train_weight_quota_gpu.py', 'quota_sampling.py', 'tpu_weight_train.py'):
    shutil.copy2(trainer.parent / name, CODE / name)

MODEL = None
for cfg in Path('/kaggle/input').rglob('config.json'):
    try:
        d = json.loads(cfg.read_text(encoding='utf-8'))
    except Exception:
        continue
    if (d.get('hidden_size'), d.get('intermediate_size'), d.get('num_hidden_layers')) == (2560, 9728, 36):
        MODEL = str(cfg.parent); break
assert MODEL, 'không thấy Qwen3-4B'
print('DATA', DATA, '| MODEL', MODEL)


In [ ]:
cmd = [sys.executable, str(CODE / 'train_weight_quota_gpu.py'),
       '--dataset', 'B', '--data-root', str(ROOT),
       '--out', '/kaggle/working/seed123_B', '--model', MODEL,
       '--seed', '123', '--tokens', '4000000', '--steps', '500',
       '--configs', 'natural,balanced,title_heavy']
print('RUN', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
out=Path('/kaggle/working/seed123_B')
for p in sorted(out.glob('hyp_*.parquet')): print(p.name, p.stat().st_size)
print((out/'summary.json').read_text())
